# Session 3 Explorer: The Three Failure Modes

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/buildLittleWorlds/level-2-course-material/blob/main/session-03/explorer.ipynb)

*Apply the three failure modes to a model from your Collection*

In class we broke models three ways: **tone deafness** (missing meaning), **emotional flattening** (oversimplifying meaning), and **anthropomorphic projection** (inventing meaning). We tested these on two models.

Now pick a model from **your Collection** and find out which failure modes hit it hardest. By the end you'll have a failure profile for your model and a draft research journal entry.

In [ ]:
!pip install -q transformers torch
from transformers import pipeline
import re
print("Setup complete!")

## Load YOUR Model

Pick a classification model from your Collection. It can be sentiment, emotion, toxicity, topic — anything that sorts input into categories.

Replace `YOUR-MODEL-NAME-HERE` with the model name from Hugging Face.

In [ ]:
# CHANGE THIS to a model from your Collection
# Examples:
#   "distilbert-base-uncased-finetuned-sst-2-english"  (2-label sentiment)
#   "cardiffnlp/twitter-roberta-base-sentiment-latest"  (3-label sentiment)
#   "j-hartmann/emotion-english-distilroberta-base"     (7 emotions)
#   "SamLowe/roberta-base-go_emotions"                  (28 emotions)
#   Any text-classification model from your Collection!

MODEL_NAME = "YOUR-MODEL-NAME-HERE"

my_model = pipeline("text-classification", model=MODEL_NAME, top_k=3)
print(f"Loaded: {MODEL_NAME}")
print("Ready to test!")

---

## Test 1: Tone Deafness

**Does your model miss meaning that's there?**

Write 3 sarcastic or ironic sentences. The words should sound positive (or negative), but the real meaning is the opposite. A human would catch the tone instantly.

In [ ]:
# CHANGE THESE to your own sarcastic/ironic sentences
tone_deafness_inputs = [
    "Replace with a sarcastic sentence that sounds positive.",
    "Replace with an ironic sentence.",
    "Replace with passive aggression.",
]

print("TEST 1: TONE DEAFNESS")
print("Does the model catch the real meaning, or read the words literally?")
print("=" * 60)
for text in tone_deafness_inputs:
    result = my_model(text)[0]
    print(f'\n"{text}"')
    for r in result[:3]:
        print(f"  {r['label']}: {r['score']:.0%}")

**Your verdict:** Did your model catch the sarcasm, or did it read the words literally?

Rate your model's tone deafness (circle one): **Bad / Moderate / Surprisingly Good**

*Notes:*



---

## Test 2: Emotional Flattening

**Does your model oversimplify complex emotions?**

Write 3 sentences where multiple emotions are tangled together. Joy and grief. Excitement and anxiety. Pride and guilt. A human would feel the complexity.

In [ ]:
# CHANGE THESE to sentences with genuinely mixed emotions
flattening_inputs = [
    "Replace with a sentence that's both happy and sad.",
    "Replace with a sentence that's exciting and terrifying.",
    "Replace with a sentence that's proud and guilty.",
]

print("TEST 2: EMOTIONAL FLATTENING")
print("Does the model capture the complexity, or flatten it to one label?")
print("=" * 60)
for text in flattening_inputs:
    result = my_model(text)[0]
    print(f'\n"{text}"')
    for r in result[:3]:
        print(f"  {r['label']}: {r['score']:.0%}")

**Your verdict:** Did the model capture any of the complexity? Look at the confidence spread — if the top label is 90%+, the model is very sure it's one thing. If it's more spread out, the model is at least "uncertain" (even if it can't express why).

Rate your model's flattening (circle one): **Severe / Moderate / Minimal**

*Notes:*



---

## Test 3: Anthropomorphic Projection

**Does your model invent emotions that aren't there?**

Write 3 sentences that describe something with **zero human emotion** — weather, geology, chemistry, physics, animal behavior. Pure facts, no feelings.

In [ ]:
# CHANGE THESE to factual, emotionless descriptions
projection_inputs = [
    "Replace with a factual sentence about weather or geology.",
    "Replace with a scientific observation.",
    "Replace with a description of an animal behavior.",
]

print("TEST 3: ANTHROPOMORPHIC PROJECTION")
print("There should be NO emotions here. Does the model invent them anyway?")
print("=" * 60)
for text in projection_inputs:
    result = my_model(text)[0]
    print(f'\n"{text}"')
    for r in result[:3]:
        print(f"  {r['label']}: {r['score']:.0%}")

**Your verdict:** Did the model assign emotions to emotionless text? How confident was it? Remember: a confident answer on text with no emotion is a **false positive**.

Rate your model's projection (circle one): **Heavy / Some / Almost None**

*Notes:*



---

## Bonus: Does Cleaning Help?

Pick one input from each test above and run it through `clean_text()`. This is the same function from class. The prediction is: cleaning won't change anything, because these are meaning problems, not noise problems.

In [ ]:
def clean_text(text):
    """Clean up messy input before the model sees it."""
    text = text.strip()
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(
        r'[\U0001F600-\U0001F64F'
        r'\U0001F300-\U0001F5FF'
        r'\U0001F680-\U0001F6FF'
        r'\U00002600-\U000026FF]+',
        ' ', text
    )
    words = text.split()
    caps_count = sum(1 for w in words if w.isupper() and len(w) > 1)
    if caps_count > 3:
        text = text.title()
    text = re.sub(r' {2,}', ' ', text).strip()
    return text


# Pick one input from each test (or replace with your own)
cleaning_tests = [
    ("Tone Deafness",    tone_deafness_inputs[0]),
    ("Flattening",       flattening_inputs[0]),
    ("Projection",       projection_inputs[0]),
]

print("CLEANING TEST")
print("=" * 60)
for label, text in cleaning_tests:
    cleaned = clean_text(text)
    r_before = my_model(text)[0][0]
    r_after = my_model(cleaned)[0][0]
    changed = "CHANGED" if r_before['label'] != r_after['label'] else "same"
    print(f"\n{label}:")
    print(f'  "{text}"')
    print(f"  Before: {r_before['label']} ({r_before['score']:.0%})")
    print(f"  After:  {r_after['label']} ({r_after['score']:.0%}) [{changed}]")

---

## Your Model's Failure Profile

Fill this in based on your tests above. This is the key takeaway from three weeks of work.

**Model tested:** *(name)*

| Failure Mode | How Bad? | Example That Shows It |
|---|---|---|
| Tone Deafness | *(bad / moderate / good)* | *(your best example)* |
| Emotional Flattening | *(severe / moderate / minimal)* | *(your best example)* |
| Anthropomorphic Projection | *(heavy / some / almost none)* | *(your best example)* |

**Does cleaning help with any of these?** *(yes / no / explain)*

**Overall:** This model is good at _____________ but bad at _____________. These failures are all **meaning problems** that classification can't solve, no matter how many labels it has.

---

## The Bigger Picture

The model you just tested is a **classification model**. It sorts inputs into categories from a fixed menu. Every model we've used in Sessions 1–3 does this.

Classification is real, useful work. But the three failure modes you just tested are **fundamental limits** of classification. They come from the structure of the task itself — choosing one label from a menu — not from any particular model being bad.

For decades, AI researchers lived in this world. They built narrow models for narrow tasks. They cleaned data, engineered features, and tuned parameters — all to make classification a little better. It worked. Spam filters, content moderation, medical screening — classification powers all of these.

But it hits a wall. And the question that led to everything you use today — ChatGPT, Claude, image generators, code assistants — was:

**What if we stopped sorting into buckets? What if we asked a model to create something new?**

That's Session 4.

---

## Research Journal Draft

Fill in the sections below. This becomes your Week 3 Research Journal entry.

---

### Week 3: What Models Can't Do

**This Week's Method:**
Adversarial testing — deliberately designing inputs to break a model, organized around three failure modes: tone deafness, emotional flattening, and anthropomorphic projection.

**How I Applied It:**
*(Which model did you test? What inputs did you design for each failure mode?)*



**What I Found:**
*(Which failure mode hit your model hardest? Were any failures surprising? Did cleaning help?)*



**The Bigger Question:**
*(These failures come from the structure of classification itself — sorting into buckets. Could a different kind of AI handle them better? What would that look like?)*



**What I Want to Try Next:**
*(What new question did this give you? Is there a failure mode that interests you more than the others?)*



---

## Collection Tasting Note

Add this to your model's entry in your Collection on Hugging Face.

---

**Model name:**

**Tone deafness (sarcasm/irony):** *(how did it do?)*

**Emotional flattening (mixed feelings):** *(how did it do?)*

**Anthropomorphic projection (emotionless text):** *(how did it do?)*

**Biggest weakness:**

**I'd trust this model with:**

**I would NOT trust this model with:**

---

*Copy this tasting note into your Collection description on Hugging Face.*